# Exploratory Data Analysis — Marketing & Sales Dataset
**Project**: Optimisation du Retour sur Investissement Marketing  
**Dataset**: `data/Dummy Data HSS.csv`  
**Target variable**: `Sales` (in millions)  
**Task**: Regression — predict Sales from TV, Radio, Social Media budgets + Influencer type

## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

DATA_PATH = '../data/Dummy Data HSS.csv'

## 1. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Pct (%)': missing_pct})
print(missing_df[missing_df['Count'] > 0])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
missing_pct[missing_pct > 0].plot(kind='bar', ax=ax, color='salmon', edgecolor='black')
ax.set_title('Missing Values per Column (%)')
ax.set_ylabel('% missing')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Conclusion: <0.3% missing in all columns — safe to impute with median

## 3. Descriptive Statistics

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std', 'min', 'max'])

In [ ]:
# Influencer category distribution
influencer_counts = df['Influencer'].value_counts()
print(influencer_counts)
print(f'\nProportion (%):\n{(influencer_counts / len(df) * 100).round(1)}')

## 4. Distribution of Numerical Features

In [ ]:
num_cols = ['TV', 'Radio', 'Social Media', 'Sales']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    # Histogram
    axes[0, i].hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'{col} — Distribution')
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Count')
    # Boxplot
    axes[1, i].boxplot(data, vert=True, patch_artist=True,
                       boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[1, i].set_title(f'{col} — Boxplot')
    axes[1, i].set_xticks([])

plt.suptitle('Feature Distributions & Outliers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Influencer Type — Sales Distribution

In [ ]:
order = df.groupby('Influencer')['Sales'].median().sort_values(ascending=False).index

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot: Sales by Influencer
sns.boxplot(data=df, x='Influencer', y='Sales', order=order, palette='Set2', ax=axes[0])
axes[0].set_title('Sales Distribution by Influencer Type')
axes[0].set_xlabel('Influencer Type')
axes[0].set_ylabel('Sales (M)')

# Count of campaigns per influencer
sns.countplot(data=df, x='Influencer', order=order, palette='Set2', ax=axes[1])
axes[1].set_title('Number of Campaigns by Influencer Type')
axes[1].set_xlabel('Influencer Type')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, square=True,
    linewidths=0.5, ax=ax
)
ax.set_title('Correlation Matrix (lower triangle)')
plt.tight_layout()
plt.show()

## 7. Budget Channels vs Sales (Scatter Plots)

In [ ]:
budget_cols = ['TV', 'Radio', 'Social Media']
influencer_palette = {'Mega': '#e41a1c', 'Macro': '#377eb8', 'Micro': '#4daf4a', 'Nano': '#ff7f00'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(budget_cols):
    for inf_type, color in influencer_palette.items():
        subset = df[df['Influencer'] == inf_type]
        axes[i].scatter(subset[col], subset['Sales'], alpha=0.35, s=15,
                        color=color, label=inf_type)
    # Trend line (all data)
    clean = df[[col, 'Sales']].dropna()
    z = np.polyfit(clean[col], clean['Sales'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(clean[col].min(), clean[col].max(), 200)
    axes[i].plot(x_line, p(x_line), 'k--', linewidth=1.5, label='Trend')

    axes[i].set_title(f'{col} Budget vs Sales')
    axes[i].set_xlabel(f'{col} (M)')
    axes[i].set_ylabel('Sales (M)')
    if i == 0:
        axes[i].legend(fontsize=8, markerscale=1.5)

plt.suptitle('Budget Channels vs Sales — colored by Influencer Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Pairplot — Full Feature Space

In [ ]:
# Sample for speed if dataset is large
sample = df.dropna().sample(min(1000, len(df)), random_state=42)

g = sns.pairplot(
    sample[['TV', 'Radio', 'Social Media', 'Sales', 'Influencer']],
    hue='Influencer',
    palette=influencer_palette,
    plot_kws={'alpha': 0.4, 's': 15},
    diag_kind='kde'
)
g.fig.suptitle('Pairplot — Budget Features & Sales by Influencer Type', y=1.02, fontsize=13)
plt.show()

## 9. Derived Features — ROI & Budget Shares

In [ ]:
df_clean = df.dropna().copy()
df_clean['Total_Budget'] = df_clean['TV'] + df_clean['Radio'] + df_clean['Social Media']
df_clean['ROI'] = df_clean['Sales'] / df_clean['Total_Budget']
df_clean['TV_share'] = df_clean['TV'] / df_clean['Total_Budget']
df_clean['Radio_share'] = df_clean['Radio'] / df_clean['Total_Budget']
df_clean['SM_share'] = df_clean['Social Media'] / df_clean['Total_Budget']

print('ROI stats:')
print(df_clean['ROI'].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROI distribution
axes[0].hist(df_clean['ROI'], bins=50, color='teal', edgecolor='white', alpha=0.8)
axes[0].set_title('ROI Distribution (Sales / Total Budget)')
axes[0].set_xlabel('ROI')
axes[0].set_ylabel('Count')

# ROI by Influencer
sns.boxplot(data=df_clean, x='Influencer', y='ROI',
            order=order, palette='Set2', ax=axes[1])
axes[1].set_title('ROI by Influencer Type')
axes[1].set_xlabel('Influencer Type')
axes[1].set_ylabel('ROI')

plt.tight_layout()
plt.show()

In [ ]:
# Budget channel shares — stacked bar by Influencer
share_means = df_clean.groupby('Influencer')[['TV_share', 'Radio_share', 'SM_share']].mean()
share_means.columns = ['TV', 'Radio', 'Social Media']

share_means.loc[order].plot(
    kind='bar', stacked=True, figsize=(8, 5),
    color=['steelblue', 'coral', 'seagreen'], edgecolor='white'
)
plt.title('Average Budget Channel Share by Influencer Type')
plt.ylabel('Share of Total Budget')
plt.xlabel('Influencer Type')
plt.xticks(rotation=0)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 10. Saturation Effect — TV Budget vs Sales (Binned)

In [ ]:
df_clean['TV_bin'] = pd.qcut(df_clean['TV'], q=10, duplicates='drop')
tv_grouped = df_clean.groupby('TV_bin', observed=True)['Sales'].agg(['mean', 'median', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(tv_grouped))
ax.bar(x, tv_grouped['mean'], alpha=0.6, color='steelblue', label='Mean Sales')
ax.errorbar(x, tv_grouped['mean'], yerr=tv_grouped['std'],
            fmt='none', color='navy', capsize=4, linewidth=1.2)
ax.plot(x, tv_grouped['median'], 'o--', color='tomato', label='Median Sales', linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in tv_grouped['TV_bin']], rotation=45, ha='right', fontsize=8)
ax.set_title('Mean & Median Sales by TV Budget Decile')
ax.set_xlabel('TV Budget Range (M)')
ax.set_ylabel('Sales (M)')
ax.legend()
plt.tight_layout()
plt.show()
# Look for: does Sales growth slow down at high TV budgets? (saturation)

## 11. Key Findings Summary

In [ ]:
print('=== EDA Summary ===')
print(f'Dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Missing values: {df.isnull().sum().sum()} total ({df.isnull().sum().sum()/df.size*100:.2f}% of all cells)')
print(f'\nInfluencer distribution:')
print(df['Influencer'].value_counts(normalize=True).mul(100).round(1).to_string())
print(f'\nCorrelation with Sales:')
print(df[num_cols].corr()['Sales'].drop('Sales').sort_values(ascending=False).round(3).to_string())
print(f'\nROI stats (Sales / Total Budget):')
print(df_clean['ROI'].describe().round(3).to_string())

---
## Next Step → `src/preprocessing.py`
Based on this EDA:
- Impute TV / Radio / Social Media nulls with **median** (skewed distributions)
- Drop rows where `Sales` is null (6 rows, it's the target)
- **OneHotEncode** `Influencer` (4 categories)
- **StandardScaler** on all numeric features (needed for Linear Regression + MLP)
- Train/test split: **80/20**, stratified if using classification later